# Reproducible two-Mac ens-svm training and custody replay
This is a reusable protocol for an **authorized local** experiment. The original two Mac CLI runs were EXECUTED; the notebook cells here are NOT_EXECUTED. No challenge rows, slide IDs, labels, or predictions may be sent to public GitHub.
**Prerequisites:** pinned dna-forge commit (recorded in evidence), local Biobase slide and derived vector ZIP archives, Python 3.9.6, NumPy 2.0.2, scikit-learn 1.6.1, SciPy 1.13.1. Use an isolated private output directory. Do not overwrite the pre-4 PM frozen files.


In [ ]:
from pathlib import Path
import hashlib, os, subprocess, sys
from zipfile import ZipFile
repo = Path.cwd()
assert (repo/'dnaforge'/'models.py').is_file()
base=Path.home()/'projects/private/biobase-20261114'
raw=base/'raw'
data=base/'dnaforge-node'
runs=base/'notebook-enssvm-runs'
print('CODE',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('DIRTY',bool(subprocess.check_output(['git','status','--porcelain'],text=True).strip()))
print('PYTHON',sys.version.split()[0])


In [ ]:
assert (raw/'biobase-challenge-slides.zip').is_file()
assert (raw/'slide-vectors-derived-20261114.zip').is_file()
sources=[(raw/'biobase-challenge-slides.zip','pack_slides/slides_manifest.json','slides/pack_slides/slides_manifest.json')]
for model in ('gigapath','hoptimus','biomedclip','dinov2'):
    member='tilepack/embeddings/'+model+'/slide-vectors.json'
    sources.append((raw/'slide-vectors-derived-20261114.zip',member,'tiles/'+member))
for archive,member,rel in sources:
    with ZipFile(archive) as z: b=z.read(member)
    dst=data/rel
    dst.parent.mkdir(parents=True,exist_ok=True)
    if dst.exists(): assert dst.read_bytes()==b, 'Existing source mismatch'
    else:
        dst.write_bytes(b)
        dst.chmod(0o600)
    print(rel,hashlib.sha256(b).hexdigest())


In [ ]:
fold_dir=repo/'custody/local'
fold_dir.mkdir(parents=True,exist_ok=True)
subprocess.run([sys.executable,'-m','dnaforge','--root',str(data),'folds','--out',str(fold_dir/'folds.json'),'--hash-out',str(fold_dir/'folds.sha256')],check=True)
expected=(repo/'custody/folds.sha256').read_text().split()[0]
observed=(fold_dir/'folds.sha256').read_text().split()[0]
assert expected==observed, 'fold-definition mismatch'
print('FOLDS_SHA',observed)


In [ ]:
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(), 'Dirty code is inadmissible for clean replay'
runs.mkdir(parents=True,exist_ok=True)
env=dict(os.environ,OPENBLAS_NUM_THREADS='1',OMP_NUM_THREADS='1')
subprocess.run([sys.executable,'-m','dnaforge','--root',str(data),'--runs',str(runs),'submit','--method','ens-svm','--alpha','3','--folds',str(fold_dir/'folds.json')],cwd=repo,env=env,check=True)
records=sorted(runs.glob('*/record.json'))
print('RECORDS',len(records))


In [ ]:
# Separate machine-specific, locally rehashed public-safe summary.
# The CLI refuses dirty code, source/output mismatches or overwritten receipts.
# Choose a NEW destination in a protected worktree before proposing a public PR.
# Example (replace placeholders): python scripts/crossnode_receipts.py summarize
#   --record /path/to/private/run/record.json
#   --node magicSTUDIObox
#   --out docs/evidence/unique_new_magicstudio_run.json
print('See docs/PI_PM_CLAUDE_RECOVERY_CROSSNODE_20261114.md')
